# Phase 1: Data Acquisition and Aggregation (XJTU Dataset)

**Goal:** Aggregate the raw high-resolution XJTU vibration data. 
The system consolidates thousands of single-minute CSV files into a unified sequence mapped per bearing. This ensures the raw signal integrity is maintained natively for the subsequent sliding window segmentation phase.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
# Define input and output paths
RAW_DATA_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets"
OUTPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\Aggregated_Raw"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Dataset Specifics
# XJTU: 25.6 kHz, ~1.28 sec snapshot (32,768 rows) sampled every 1 minute.
ROWS_PER_FILE = 32768

print(f"Raw Data Path: {os.path.abspath(RAW_DATA_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")

In [ ]:
def aggregate_bearing_data():
    """
    Iterates through specific condition directories ('35Hz12kN', '37.5Hz11kN', '40Hz10kN')
    and their respective raw bearing files. Consolidates the high-resolution vibration 
    arrays into a single sequential dataframe per bearing.
    """
    if not os.path.exists(RAW_DATA_PATH):
        print(f"Warning: Raw data path '{RAW_DATA_PATH}' does not exist.")
        return

    # Specific Target Folders Based on Final Blueprint
    target_conditions = ['35Hz12kN', '37.5Hz11kN', '40Hz10kN']
    
    summary_info = []
    last_df = None
    
    for condition in target_conditions:
        cond_path = os.path.join(RAW_DATA_PATH, condition)
        if not os.path.exists(cond_path):
            print(f"Skipping {condition}, folder not found.")
            continue
            
        bearing_folders = [f for f in os.listdir(cond_path) if os.path.isdir(os.path.join(cond_path, f))]
        
        for bearing in tqdm(bearing_folders, desc=f"Processing {condition}"):
            bearing_path = os.path.join(cond_path, bearing)
            csv_files = sorted(glob.glob(os.path.join(bearing_path, "*.csv")))
            
            if not csv_files:
                continue
                
            bearing_data_list = []
            
            for minute_idx, file in enumerate(csv_files):
                try:
                    df_min = pd.read_csv(file)
                except Exception as e:
                    print(f"Error reading {file}: {e}")
                    continue
                
                # Extract raw signals based on standard XJTU structure
                if 'Horizontal_vibration_signals' in df_min.columns:
                    h_sig = df_min['Horizontal_vibration_signals'].values
                    v_sig = df_min['Vertical_vibration_signals'].values
                else:
                    h_sig = df_min.iloc[:, 0].values
                    v_sig = df_min.iloc[:, 1].values if df_min.shape[1] > 1 else np.zeros_like(h_sig)
                
                # Store the full high-resolution array sequentially
                sequence_record = {
                    'Minute': minute_idx + 1,
                    'H_Signal': h_sig,
                    'V_Signal': v_sig
                }
                bearing_data_list.append(sequence_record)
            
            if bearing_data_list:
                final_df = pd.DataFrame(bearing_data_list)
                
                cond_out_path = os.path.join(OUTPUT_PATH, condition)
                os.makedirs(cond_out_path, exist_ok=True)
                
                output_filename = os.path.join(cond_out_path, f"{bearing}_aggregated.pkl")
                final_df.to_pickle(output_filename)
                
                file_size_mb = os.path.getsize(output_filename) / (1024 * 1024)
                summary_info.append({
                    "Condition": condition, 
                    "Bearing": bearing, 
                    "Total_Sequences": len(final_df), 
                    "Size_MB": round(file_size_mb, 2)
                })
                last_df = final_df
                print(f"Saved: {output_filename} (Sequences: {len(final_df)})")

    print("\n--- Aggregation Summary ---")
    if summary_info:
        summary_df = pd.DataFrame(summary_info)
        print(summary_df.to_string(index=False))
    else:
        print("No data processed.")
        
    if last_df is not None:
        print(f"\n--- Sample Data Structure (First 5 Sequences) ---")
        print(last_df.head(5))

# Execute the pipeline
if __name__ == "__main__":
    aggregate_bearing_data()